# מעבדה 08 — אנטרופיה וריבוי מצבים

במעבדה הזו תספרו מיקרו-מצבים ישירות, תצפו כיצד אנטרופיית בולצמן מתגלה מתוך הספירה הזו,
ותשתמשו באותה ספירה כדי לשבור שתי אינטואיציות נפוצות לגבי אנטרופיה.

עבדו לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, כתבו את הניבוי בתא המיועד לכך **לפני**
הרצת התא הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | $N$ עצמים ניתנים להבחנה, כל אחד באחד משני מצבים |
| **דינמיקה** | בגרסה הנדגמת, עצם אחד שנבחר באקראי מחליף מצב בכל צעד |
| **גבול** | סגור; $N$ קבוע |
| **צבר** | מיקרו-קנוני במובן שבו כל מיקרו-מצב שווה-הסתברות — הנחת היסוד שהמודול הזה בוחן |
| **מוזנח** | אינטראקציות בין העצמים, הפרשי אנרגיה בין שני המצבים |
| **תקף כאשר** | שני המצבים שקולים אנרגטית והעצמים בלתי תלויים |
| **אופני כישלון** | מערכות בעלות אינטראקציה, או מצבים בעלי אנרגיות שונות, שבהם גורם בולצמן משתלט |

כל הפיזיקה נמצאת ב-`thermolab.multiplicity` — פתחו וקראו אותה. שום דבר בקורס הזה אינו
מוסתר בתוך תשתית תוכנה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import gammaln

from thermolab import multiplicity
from thermolab.constants import K_B
from thermolab.validation import relative_error, scaling_exponent

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"k_B = {K_B:.6e} J/K")

## חלק 1 — ספירת מיקרו-מצבים ביד

התחילו קטן מספיק כדי לספור ישירות. עשרה עצמים ניתנים להבחנה, כל אחד באופן בלתי תלוי
באחד משני מצבים, נותנים בסך הכול $2^{10} = 1024$ מיקרו-מצבים.

In [ ]:
N_SMALL = 10
counts = np.arange(N_SMALL + 1)
omegas = np.array([multiplicity.multiplicity(N_SMALL, int(n)) for n in counts])

for n, omega in zip(counts, omegas, strict=True):
    print(f"n = {n:2d}   Omega(10, n) = {omega:6.0f}   P(n) = {omega / 2**N_SMALL:.5f}")

print(f"\nsum of Omega(10, n) = {omegas.sum():.0f}   (2^10 = {2**N_SMALL})")

plt.figure(figsize=(5.5, 3.5))
plt.bar(counts, omegas, color="#2563eb")
plt.xlabel("n (objects in state 1)")
plt.ylabel(r"$\Omega(10, n)$")
plt.title("Multiplicity of every macrostate, N = 10")
plt.tight_layout()
plt.show()

המאקרו-מצב $n=5$ לבדו אחראי לכרבע מכל $1024$ המיקרו-מצבים, בעוד הקיצונים המסודרים
לחלוטין $n=0$ ו-$n=10$ אחראים כל אחד לכאחד מתוך אלף. הפער הזה הוא כל תוכן המודול, נראה
לעין כבר ב-$N=10$.

### לנבא

לפני הרצת התא הבא, רשמו מה אתם מצפים שיקרה לרוחב ה*יחסי* של השיא הזה, $\sigma/(N/2)$,
ככל ש-$N$ גדל מ-$100$ ל-$10{,}000$ ועד $1{,}000{,}000$. האם מיקום השיא יזוז? האם רוחבו
היחסי יצטמצם, יגדל, או יישאר זהה?

**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*

In [ ]:
sizes = np.array([100, 1_000, 10_000, 100_000, 1_000_000])
widths = np.array([multiplicity.peak_relative_width(int(n)) for n in sizes])
exponent = scaling_exponent(sizes, widths)

for n, w in zip(sizes, widths, strict=True):
    print(f"N = {n:>9}   sigma/(N/2) = {w:.6f}   1/sqrt(N) = {1 / np.sqrt(n):.6f}")
print(f"\nfitted exponent = {exponent:.6f}   (theory: -0.5)")

plt.figure(figsize=(5.5, 4))
plt.loglog(sizes, widths, "o", label="peak_relative_width(N)")
plt.loglog(sizes, 1 / np.sqrt(sizes), "-", label=r"$N^{-1/2}$")
plt.xlabel("N")
plt.ylabel(r"$\sigma / (N/2)$")
plt.title(f"relative peak width (fitted slope {exponent:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

השיא לעולם אינו זז — הוא תמיד יושב ב-$n=N/2$. מה שמשתנה הוא באיזו חדות כל השאר נדחק
החוצה, וזה דועך כמו $N^{-1/2}$: אותו חוק ששלט בפלוקטואציות הלחץ במודול 04, כאן נגזר
משום דבר מלבד קומבינטוריקה.

## חלק 2 — אנטרופיית בולצמן

$S = k_B \ln \Omega$. חשבו אותה על פני כל מאקרו-מצב של מערכת בגודל קבוע, וצפו בה משיאה,
באופן סימטרי, בחלוקה השווה, ומתאפסת בדיוק בקיצונים המסודרים לחלוטין.

In [ ]:
N_ENTROPY = 200
n_values = np.arange(N_ENTROPY + 1)
entropies_over_kb = np.array([multiplicity.log_multiplicity(N_ENTROPY, int(n)) for n in n_values])

plt.figure(figsize=(6, 4))
plt.plot(n_values, entropies_over_kb, color="#2563eb")
plt.axvline(N_ENTROPY / 2, color="crimson", ls="--", lw=1.0)
plt.xlabel("n")
plt.ylabel(r"$S(N, n) / k_B$")
plt.title(f"Entropy across every macrostate, N = {N_ENTROPY}")
plt.tight_layout()
plt.show()

print(f"S(N, 0)      = {multiplicity.entropy(N_ENTROPY, 0):.3e} J/K   (Omega = 1, fully ordered)")
print(f"S(N, N/2)/kB = {multiplicity.log_multiplicity(N_ENTROPY, N_ENTROPY // 2):.4f}")

## חלק 3 — קירוב סטירלינג ואקסטנסיביות

$\ln N!$ עבור $N$ מאקרוסקופי לעולם אינו מחושב ישירות; כל מה שלמעלה נשען על קירוב
סטירלינג. מדדו את השגיאה שלו מול הערך המדויק, ומדדו עד כמה האנטרופיה רחוקה מלהיות
אקסטנסיבית באופן מדויק עבור $N$ סופי.

In [ ]:
print("Stirling's approximation to ln(N!):")
for n in (10, 100, 1_000, 10_000):
    exact = float(gammaln(n + 1))
    order0 = multiplicity.stirling_log_factorial(n, order=0)
    order1 = multiplicity.stirling_log_factorial(n, order=1)
    print(
        f"  n={n:>6}   exact={exact:12.4f}   |exact-2term|={exact - order0:8.4f}   "
        f"|exact-3term|={exact - order1:10.6f}   1/(12n)={1 / (12 * n):.6f}"
    )

print("\nExtensivity discrepancy: S(2N, N) vs 2 S(N, N/2)")
for n in (200, 2_000, 20_000, 200_000):
    a = multiplicity.entropy(2 * n, n)
    b = 2.0 * multiplicity.entropy(n, n // 2)
    predicted = (0.5 * np.log(np.pi * n) - np.log(2.0)) / (2.0 * n * np.log(2.0))
    print(f"  N={n:>7}   relative_error={relative_error(a, b):.6e}   predicted={predicted:.6e}")

השגיאה ה*מוחלטת* של נוסחת סטירלינג בעלת שני האיברים גדלה לאט עם $N$ (חסר לה האיבר
$\tfrac12\ln(2\pi N)$); שגיאת הצורה בעלת שלושת האיברים מצטמצמת כמו $1/(12N)$, בהתאמה
לאיבר הבא בטור האסימפטוטי. הפער באקסטנסיביות מצטמצם באותו אופן: האנטרופיה אקסטנסיבית רק
בגבול, לעולם לא באופן מדויק עבור $N$ סופי.

## חלק 4 — הגבול הגאוסי

פיתוח $\ln\Omega(N,n)$ עד לסדר שני סביב $n=N/2$ נותן התפלגות גאוסית. השוו אותה לתוצאה
הקומבינטורית המדויקת בקרבת השיא, בגודל גדול מספיק כדי שמשפט הגבול המרכזי ייכנס לתוקף.

In [ ]:
N_GAUSS = 4000
gauss_counts = np.arange(N_GAUSS // 2 - 300, N_GAUSS // 2 + 301)
exact = np.exp(multiplicity.log_multiplicity_array(N_GAUSS, gauss_counts) - N_GAUSS * np.log(2.0))
gaussian = multiplicity.gaussian_multiplicity_fraction(N_GAUSS, gauss_counts)

max_rel_error = np.max(np.abs(exact - gaussian)) / np.max(exact)
print(f"max relative error near the peak (N = {N_GAUSS}): {max_rel_error:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(gauss_counts, exact, label="exact (binomial)", lw=2, color="#2563eb")
plt.plot(gauss_counts, gaussian, "--", label="Gaussian approximation", lw=1.6, color="crimson")
plt.xlabel("n")
plt.ylabel(r"$\Omega(N,n)/2^N$")
plt.title(f"Central limit theorem, N = {N_GAUSS}")
plt.legend()
plt.tight_layout()
plt.show()

## חלק 5 — כד ארנפסט

כאשר מתחילים עם כל העצמים בצד אחד, צפו באכלוס נסחף אל החלוקה השווה ונשאר שם — לא משום
שאסור לו לעזוב, אלא משום שכה מעטים מהמיקרו-מצבים הנגישים יושבים במקום אחר כלשהו.

In [ ]:
N_URN = 200
occupancy = multiplicity.sample_two_box(N_URN, n_steps=6000, rng=rng)

plt.figure(figsize=(7, 3.5))
plt.plot(occupancy, lw=0.8, color="#2563eb")
plt.axhline(N_URN / 2, color="crimson", ls="--", lw=1.2)
plt.xlabel("step")
plt.ylabel("box A occupancy")
plt.title(f"Ehrenfest urn, N = {N_URN}, started fully in box A")
plt.tight_layout()
plt.show()

late = occupancy[len(occupancy) // 2 :]
print(f"late-time mean fraction   = {late.mean() / N_URN:.4f}   (expect close to 0.5)")
print(f"late-time spread fraction = {late.std() / N_URN:.4f}")

In [ ]:
n_realisations = 24
late_fractions = []
for _ in range(n_realisations):
    trajectory = multiplicity.sample_two_box(N_URN, n_steps=4000, rng=rng)
    late_fractions.append(trajectory[len(trajectory) // 2 :].mean() / N_URN)
late_fractions = np.array(late_fractions)

predicted_sigma = 1.0 / (2 * np.sqrt(N_URN))
print(
    f"measured std of late-time fraction across {n_realisations} runs: "
    f"{late_fractions.std(ddof=1):.4f}"
)
print(f"predicted sigma_n/N = 1/(2 sqrt(N))                          : {predicted_sigma:.4f}")

plt.figure(figsize=(5.5, 3.5))
plt.hist(late_fractions, bins=8, color="#2563eb", edgecolor="white")
plt.axvline(0.5, color="crimson", ls="--")
plt.xlabel("late-time occupancy fraction")
plt.ylabel("count")
plt.title("Spread across independent Ehrenfest runs")
plt.tight_layout()
plt.show()

## חלק 6 — הפרכת "אנטרופיה היא אי-סדר"

גביש הבנוי מאיזוטופ יחיד וגביש הבנוי מתערובת אקראית של 50/50 בין שני איזוטופים של אותו
יסוד הם, על אותו סריג, זהים מבחינה חזותית ומבנית — אותה צפיפות, אותו מבנה, אותו מראה
תחת כל מיקרוסקופ רגיל. חשבו את האנטרופיות שלהם ישירות במקום לסמוך על איך שהם נראים.

In [ ]:
N_SITES = 1000
S_pure = multiplicity.entropy(N_SITES, 0)
S_mixed = multiplicity.entropy(N_SITES, N_SITES // 2)

print(
    f"pure crystal      (n=0)   : Omega = {multiplicity.multiplicity(N_SITES, 0):.0f}"
    f"   S = {S_pure:.3e} J/K"
)
print(
    f"50/50 isotope mix (n=N/2) : ln Omega = "
    f"{multiplicity.log_multiplicity(N_SITES, N_SITES // 2):.2f}   S = {S_mixed:.3e} J/K"
)
print(f"\nThe mixed crystal has {S_mixed - S_pure:.3e} J/K more entropy than the pure crystal,")
print("despite looking visually identical to it.")

assert S_pure == 0.0
assert S_mixed > S_pure

שום דבר בגביש המעורב אינו נראה "בלתי מסודר" יותר מהטהור. האנטרופיה שלו גדולה יותר
משום שהאנטרופיה סופרת מיקרו-מצבים נגישים, ויש דרכים רבות באופן אסטרונומי לפזר שני
איזוטופים על פני סריג תוך הפקת אותו מאקרו-מצב נראה לעין בכל פעם.

## חלק 7 — הפרכת "האנטרופיה של כל תת-מערכת חייבת לעלות"

שתי תת-מערכות דו-מצביות בלתי תלויות, $L$ ו-$R$, הן חלק ממערכת מבודדת גדולה אחת. מכיוון
שהן בלתי תלויות, $\Omega_{LR} = \Omega_L \, \Omega_R$, כך שהאנטרופיות שלהן מצטרפות
באופן מדויק: $S_{LR} = S_L + S_R$. שום דבר באדיטיביות הזו אינו מחייב שכל איבר ינוע באותו
כיוון.

In [ ]:
N_SUB = 300
ln_omega_L_before = multiplicity.log_multiplicity(N_SUB, 150)  # L starts at its own peak
ln_omega_L_after = multiplicity.log_multiplicity(N_SUB, 145)  # nudged away from its peak
ln_omega_R_before = multiplicity.log_multiplicity(N_SUB, 280)  # R starts far from its peak
ln_omega_R_after = multiplicity.log_multiplicity(N_SUB, 150)  # R relaxes to its own peak

# S = k_B ln Omega, so a change in ln Omega is a change in S measured in units of k_B.
delta_S_L = ln_omega_L_after - ln_omega_L_before
delta_S_R = ln_omega_R_after - ln_omega_R_before
delta_S_total = delta_S_L + delta_S_R  # exact, because L and R are independent

print(f"Delta S_L / k_B     = {delta_S_L:+.4f}   (subsystem L: entropy DECREASES)")
print(f"Delta S_R / k_B     = {delta_S_R:+.4f}   (subsystem R: entropy increases)")
print(f"Delta S_total / k_B = {delta_S_total:+.4f}   (isolated total: entropy increases)")

assert delta_S_L < 0
assert delta_S_total > 0
print("\nA subsystem's entropy fell while the isolated total's entropy rose.")

## חלק 8 — בדיקות אוטומטיות

סימולציה שלא בדקתם היא תמונה, לא ראיה. אלה בדיוק אותן טענות הרצות בחבילת הבדיקות של
הפרויקט.

In [ ]:
from math import comb, isclose

# 1. Exact combinatorics for small systems.
for n in (1, 2, 10, 25):
    for k in range(n + 1):
        assert isclose(multiplicity.multiplicity(n, k), comb(n, k), rel_tol=1e-9)

# 2. The peak sits at the even split.
peak_counts = np.arange(0, 101)
peak_values = multiplicity.log_multiplicity_array(100, peak_counts)
assert int(peak_counts[np.argmax(peak_values)]) == 50

# 3. Stirling's error bound.
for n in (10, 100, 1000):
    exact = float(gammaln(n + 1))
    approx = multiplicity.stirling_log_factorial(n, order=1)
    assert abs(exact - approx) < 1.0 / (12.0 * n) * 1.01

# 4. The Gaussian limit stays under 1% near the peak at N=4000 (measured in Part 4).
assert max_rel_error < 0.01

# 5. The Ehrenfest urn settles near the even split and stays there (measured in Part 5).
assert abs(late.mean() / N_URN - 0.5) < 0.05

# 6. Entropy of a fully ordered macrostate is exactly zero.
assert multiplicity.entropy(500, 0) == 0.0
assert multiplicity.entropy(500, 500) == 0.0

print("all checks passed")

## חלק 9 — לחקור בעצמכם

הזיזו את $N$ וצפו בהתפלגות ההסתברות על פני המאקרו-מצבים מתחדדת סביב החלוקה השווה. מצאו
את ה-$N$ הקטן ביותר שעבורו הייתם קוראים לחלוקה השווה "ודאית למעשה", ואמרו באיזה קנה
מידה השתמשתם כדי להכריע.

In [ ]:
import ipywidgets as widgets


def explore(n_objects=100):
    counts = np.arange(n_objects + 1)
    log_omega = multiplicity.log_multiplicity_array(n_objects, counts)
    prob = np.exp(log_omega - n_objects * np.log(2.0))

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(counts, prob, color="#2563eb", width=1.0)
    ax.axvline(n_objects / 2, color="crimson", ls="--")
    ax.set_xlabel("n")
    ax.set_ylabel("P(n)")
    ax.set_title(
        f"N = {n_objects}   relative peak width = {multiplicity.peak_relative_width(n_objects):.4f}"
    )
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    n_objects=widgets.IntSlider(min=4, max=400, step=2, value=100, description="N"),
);

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית. אותן שאלות, בתוספת הסברים כתובים לכל
אפשרות, נמצאות בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "08-multiplicity.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהיגיון שלכם?
2. דוגמת גביש-האיזוטופים הראתה שני מאקרו-מצבים זהים חזותית עם אנטרופיות שונות מאוד.
   נסחו, במילים שלכם, מה האנטרופיה באמת מודדת אם לא אי-סדר חזותי.
3. הסבירו, בלי משוואות, כיצד האנטרופיה של תת-מערכת יכולה לרדת בעוד האנטרופיה של המערכת
   המבודדת הכוללת שאליה היא שייכת עולה.

**התשובות שלכם:**

1.
2.
3.